# Two trainable baselines for wave classification

by Andrés Muñoz-Jaramillo (template), adapted for wave classification by Diego

This notebook defines two simple, trainable baselines for the EUV wave classification
task, to be evaluated on the validation set before fine-tuning the full Surya backbone:

1. **Constant-probability baseline** — a single learned scalar that predicts the class
   base rate (wave vs. no wave), ignoring the image entirely.
2. **Running-difference baseline** — AIA193(now) - AIA193(one cadence step earlier),
   mean-pooled 32x32, then a logistic regression.

It focuses on the concept of defining a PyTorch model, a PyTorch lightning training loop
and the definition of metrics of performance.

This notebook assumes familiarity with the concepts of datasets and dataloaders contained
in **_0_dataset_dataloader_template_diego.ipynb_**

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.   

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
from pathlib import Path
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# The wandb module itself, not just Lightning's WandbLogger wrapper. Two things below need
# it: ending one baseline's run before the next WandbLogger is built (otherwise Lightning
# reuses the live run and both baselines land on one set of axes), and attaching the
# per-epoch figures as run media.
import wandb

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks

torch.set_float32_matmul_precision('medium')


## Load configuration

Surya was designed to read a configuration file that defines many aspects of the model
including the data it uses we use this config file to set default values that do not
need to be modified, but also to define values specific to our downstream application

In [3]:
# The config is the single source of truth. load_wave_config() parses it into a typed
# object, exactly as the training script 3_finetune_template_1D.py does, so the same YAML
# behaves identically here and in production. Notebook 0 walks through what it contains.
from downstream_apps.test.configs import load_wave_config

cfg = load_wave_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


Loaded config for job: wave_classification


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.


In [4]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# The linear baseline needs no backbone, so skip the 1.8 GB weights.
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


Loaded scalers for 13 channels.


## Define Downstream (DS) dataset

This child class (`waveDSDataset`) takes as input all expected HelioFM parameters, plus
additional parameters relevant to the downstream application. Here we focus in particular
on the DS index (the wave/no-wave catalog) and the parameters necessary to combine it with
the HelioFM index.

The label is the binary `class` column (`"wave"` / `"no wave"`), encoded to `1.0` / `0.0`
by `wave_common.wave_label_transform`.

The train / validation / test datasets are built by **one** helper,
`experiments/wave_common.py:build_wave_datasets()`, which is also what
`2_finetune_template_1D_diego.ipynb` and `4_finetune_wave_1D.py` call. That is deliberate:
the baseline in this notebook and the Surya fine-tune in notebook 2 must be trained and
scored on the *same events*, or the comparison between them means nothing. Sharing the
construction is what guarantees it, rather than two cells that happen to agree today.


In [5]:
from downstream_apps.test.datasets.wave_dataset import waveDSDataset

In [6]:
## build_helio_dataloaders() constructs the train and validation datasets and wraps them
# in DataLoaders. It fills in every generic argument (channels, temporal sampling, S3
# access, worker settings) from the config — see notebook 0 for what that block looks
# like written out. Only the flare-specific arguments are passed here, which is exactly
# the list you replace when you fork the template.
#
# It also handles two details that are easy to get wrong by hand: the validation set gets
# phase="val" (no random channel masking or flips), and only the training loader shuffles.
from workshop_infrastructure.datasets.builders import build_helio_dataloaders

train_data_loader, val_data_loader = build_helio_dataloaders(
    cfg,
    waveDSDataset,
    scalers=scalers,
    num_workers=4,          # fewer workers than the script: notebooks start faster
    #### Downstream (DS) specific parameters
    return_surya_stack=True,
    max_number_of_samples=10,
    ds_wave_index_path=cfg.data.wave_index_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
    ds_class_column=cfg.data.ds_class_column,
)

batch_size = cfg.batch_size
print(f"train: {len(train_data_loader.dataset)} samples | "
      f"val: {len(val_data_loader.dataset)} samples | batch_size: {batch_size}")

train: 10 samples | val: 10 samples | batch_size: 2


Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.


In [7]:
print("kernel alive check2")

kernel alive check2


In [11]:
import importlib
from downstream_apps.test.experiments import wave_common as wc
wc = importlib.reload(wc)
BASELINE_CHANNELS_ONLY = True
if BASELINE_CHANNELS_ONLY:
    cfg.data.channels = ["aia193"]

TRAIN_N = 100       # samples (192 events). None keeps every complete pair (~1210).
SUBSET_SEED = 42     # fixes WHICH events; shared with notebook 2 and the training script
NUM_WORKERS = 6
PREFETCH_FACTOR = 1
PERSISTENT_WORKERS = True#False

train_dataset, val_dataset, test_dataset = wc.build_wave_datasets(
    cfg, scalers, include_test=True, in_memory=True,
)
for split_name, dataset, n in [("train", train_dataset, TRAIN_N),
                               ("val", val_dataset, None),
                               ("test", test_dataset, None)]:
    info = wc.event_stratified_subset(dataset, n_samples=n, seed=SUBSET_SEED)
    print(f"{split_name:>5}: {info.describe()}")

# shuffle=True only for train; drop_last=True only for train (a partial batch adds noise to
# the gradient, but dropping val samples corrupts the reported metric).
train_data_loader = wc.build_wave_loader(
    train_dataset, batch_size=cfg.batch_size, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, seed=SUBSET_SEED, shuffle=True, drop_last=True,
    persistent_workers=PERSISTENT_WORKERS,
)
val_data_loader = wc.build_wave_loader(
    val_dataset, batch_size=cfg.batch_size, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, seed=SUBSET_SEED, shuffle=False, drop_last=False,
    persistent_workers=PERSISTENT_WORKERS,
)

batch_size = cfg.batch_size
worst_case_gb = NUM_WORKERS * PREFETCH_FACTOR * batch_size * 0.134 * len(cfg.data.channels)

train: 100 samples / 50 events (50 wave, 50 no wave) from 596 complete pairs available; dropped 18 unpaired sample(s)
  val: 24 samples / 12 events (12 wave, 12 no wave) from 12 complete pairs available; dropped 1 unpaired sample(s)
 test: 48 samples / 24 events (24 wave, 24 no wave) from 24 complete pairs available; dropped 2 unpaired sample(s)


In [12]:
# Inspect a single batch to confirm shapes before training. Dimension mismatches are the
# dominant source of error in this kind of work, so this is worth a cell of its own.
batch = next(iter(train_data_loader))
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in batch.items()})

# Derive model dimensions from the batch/config rather than hardcoding them.
#
# aia193_index MUST come from cfg.data.channels, never be written as a literal: it is 3 in
# the 13-channel configuration and 0 once the cell above narrows channels to ["aia193"].
# A hardcoded 3 would silently read a non-existent channel (IndexError) or, in a 13-channel
# run with a reordered list, train the baseline on the wrong wavelength.
n_channels = len(cfg.data.channels)
aia193_index = cfg.data.channels.index("aia193")
img_size = batch["ts"].shape[-1]     # reported only; the model no longer needs it
print(f"n_channels={n_channels} | aia193_index={aia193_index} | img_size={img_size}")

# ts is (B, C, T, H, W). T must be 2: the running difference is frame[-1] - frame[0], so a
# single timestep would make the feature identically zero.
assert batch["ts"].shape[2] == 2, (
    f"expected 2 input timesteps, got {batch['ts'].shape[2]} — check "
    f"data.time_delta_input_minutes and model.time_embedding.time_dim in the config"
)


{'ts': (2, 1, 2, 4096, 4096), 'time_delta_input': (2, 2), 'forecast': (2,), 'ds_index': 'list'}
n_channels=1 | aia193_index=0 | img_size=4096


## Define simple baseline models

Defining a simple baseline is important to understand what value is bringing the AI model
to the problem.

It is always very good to have a very simple baseline model. Ideally one that cannot
overfit the data. This is a very good way of really measuring the value added of complex
models. Classical machine learning excels here:

- Regressions and logistic regressions.
- Climatological averages.
- Persistance.
- Simple transformations.

Simple models avoid excesively optimistic assessments of the capatiblities of complex
models and for many problems are actually remarkably hard to beat.

Here we define two such baselines:

1. `ConstantProbabilityModel` — a single trainable scalar (logit) that ignores the input
   entirely and learns the wave/no-wave base rate.
2. `RunningDifferenceLogisticModel` — AIA193(now) - AIA193(one cadence step earlier),
   mean-pooled with a 32x32 kernel, then a linear layer (logistic regression via
   `BCEWithLogitsLoss`).

As with the dataset, we import both from a module so they can be reused in training
scripts later on.

In [13]:
from downstream_apps.test.models.simple_baseline import (
    ConstantProbabilityModel,
    RunningDifferenceLogisticModel,
    destandardize_channels,
    # The feature the running-difference baseline is built on, exposed as a plain function
    # so the diagnostic cell at the end of this notebook can score the validation set
    # directly — without it, a 1-feature model's result is impossible to interpret.
    mean_abs_pooled_running_difference,
)


We can now test that both models manipulate a batch as expected and return a wave
probability logit.

`ConstantProbabilityModel` needs no configuration — it ignores the image, and has exactly
**one** parameter. `RunningDifferenceLogisticModel` needs only the AIA193 channel index
(pulled from the config, never hardcoded) and the pooling kernel; it has exactly **two**
parameters, a weight and a bias on a single scalar feature.

That parameter count is the point of the whole notebook. A two-parameter model that is
monotone in one feature is a *threshold detector*: it cannot memorize the training set, and
its AUROC is a fixed property of the feature rather than something training discovers. Only
the loss and the calibration are learned. So if Surya beats this, the gain is real.

Note that since neither model has been trained yet and was initialized randomly, the
output here has no real meaning. It only acts as a test that the forward pass doesn't have
dimension problems — dimension problems are the dominant source of error in this kind of
work.


In [14]:
POOL_KERNEL = 32   # mean-pool kernel for the running difference; 4096 -> 128x128
aia193_index = cfg.data.channels.index("aia193")
# Baseline 1: a single learned logit, independent of the input.
#model_constant = ConstantProbabilityModel()

# Baseline 2: AIA193 running difference, mean-pooled, magnitude-averaged, then a logistic
# regression on that one scalar. It expects 'ts' in signum-log space — destandardize_channels()
# is wired in as the Lightning module's preprocess_fn below, so forward() always sees it.
#
# NOTE: no img_size argument. The model pools with a fixed kernel and then averages over
# whatever spatial extent remains, so it is resolution-independent by construction and the
# frame size is not something it needs to be told.
model_running_diff = RunningDifferenceLogisticModel(
    channel_index=aia193_index,
    pool_kernel=POOL_KERNEL,
)

for name, m in [("running_diff", model_running_diff)]:
    n_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:>13}: {n_trainable} trainable parameter(s)")


 running_diff: 6 trainable parameter(s)


Now we can pass a batch to each model to confirm it returns one logit per sample. Note
that our outputs have the size of our batch.

In [15]:
batch = next(iter(train_data_loader))

# RunningDifferenceLogisticModel works in signum-log space, not normalized space.
# destandardize_channels undoes the per-channel z-score but KEEPS the log compression —
# raw DN values span too many orders of magnitude to be good features for one linear layer.
# (For true physical units, use train_dataset.inverse_transform_data() instead. See the
#  "THE THREE SPACES" block in workshop_infrastructure/datasets/helio.py.)
batch_logspace = destandardize_channels(batch, channel_order=cfg.data.channels, scalers=scalers)

#output_constant = model_constant.forward(batch)[:, 0]              # ignores 'ts' entirely
output_running_diff = model_running_diff.forward(batch_logspace)[:, 0]

#print("constant baseline output (logits):", output_constant)
print("running-difference baseline output (logits):", output_running_diff)


running-difference baseline output (logits): tensor([-0.1710, -0.1689], grad_fn=<SelectBackward0>)


## Define your metrics

Metrics are a very important part of training AI models. They provide your models with
the quantitification of error, which in turn shifts the weights towards better
pefrorming models. They also provide a way for you to monitor performance, identify
overfitting, and quantify value added.

We now initialize the metrics class which allows you to control what metrics do you want
to use as "loss" (i.e. the metrics that backpropagate through your model) and which ones
for monitoring performance. `WaveMetrics` has been converted for this task to binary
classification: BCE-with-logits for the loss, accuracy + AUROC for reporting. As with
other components, this takes the form of a loaded module that can be later used in a
training script.

In [16]:
from downstream_apps.test.metrics.template_metrics import WaveMetrics

In [17]:
def make_metrics() -> dict:
    """Build a fresh dict of WaveMetrics instances.

    Each of the two baselines below needs its own dict: the accuracy/AUROC torchmetrics
    objects inside WaveMetrics accumulate internal state across calls, so sharing one
    instance between two independently-trained models would mix their statistics.
    """
    return {
        'train_loss': WaveMetrics("train_loss"),
        # val_loss is the quantity logged as "val_loss" and used to pick the best checkpoint.
        # It defaults to the same BCE as train_loss — override WaveMetrics.val_loss to change it.
        'val_loss': WaveMetrics("val_loss"),
        'train_metrics': WaveMetrics("train_metrics"),
        # Reported only: val_metrics do NOT influence checkpoint selection.
        'val_metrics': WaveMetrics("val_metrics"),
    }

demo_metrics = make_metrics()

Now they can be evaluated on a model's output and our ground truth. First the loss that
actually will backpropagate — binary cross-entropy with logits — demonstrated here on the
running-difference baseline's output.

In [18]:
demo_metrics["train_loss"](output_running_diff, batch["forecast"])

({'bce': tensor(0.7817, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)}, [1])

Then a training evaluation that will not backpropagate and inform our model, but that we
can keep an eye on. Note that reporting lots of metrics during training will slow the
training process. I'm including it here as an example, but oftentimes it is better to put
the diagnostics only in the validation evaluation metrics.

Here we are calculating accuracy (fraction correctly classified at a 0.5 probability
threshold). Since torchmetrics' binary metrics auto-apply sigmoid to logit inputs, this
works directly on the raw model output.

In [19]:
demo_metrics["train_metrics"](output_running_diff, batch["forecast"])

({'accuracy': tensor(0.)}, [1])

In the validation evaluation metrics we report both accuracy and AUROC.

In [20]:
demo_metrics["val_metrics"](output_running_diff, batch["forecast"])

/home/jovyan/envs/surya_WS/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)


({'accuracy': tensor(0.), 'auroc': tensor(0.)}, [1, 1])

## Define your PyTorch ligthning module

In this workshop we will use PyTorch lightning to train our models.  PyTorch lighting reduces the amount of code required to implement a training loop in comparison to PyTorch (at the expense of control and versatility).  

Opening the WaveLightningModule shows a simple Lightning model implementation.  It consists of:

- An initialization of the class (metrics, model, and learning rate).
- The forward code that runs evaluation of the model.
- Training and validation steps.
- Configuration of optimizers.

In [21]:
from downstream_apps.test.lightning_modules.pl_simple_baseline import WaveLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix your random seeds so that your training exercise is reproducible.    

In [22]:
L.seed_everything(42, workers=True)

Seed set to 42


42

## Intialize Lightning modules

Now we properly initialize one Lightning module per baseline, each with its own metrics
dict (see `make_metrics()` above) so their statistics don't mix.

In [23]:
from functools import partial

# ---------------------------------------------------------------------------
# Learning rate. NOT cfg.learning_rate.
#
# cfg.learning_rate is 0.01, which was tuned for the earlier 16,385-parameter version of
# this baseline. It is far too small for the 2-parameter model. The feature
# mean(|pool32(now - prev)|) is of order 0.1, so separating the classes needs a weight of
# order 10-100. Adam moves a parameter by roughly `lr` per step, so at lr=0.01 that is
# ~3000 steps of pure travel before the model is even in the right range, and the run ends
# looking flat and uninformative.
#
# 0.1 reaches weight ~30 in ~300 steps — under two epochs at 192 steps/epoch — and leaves
# the residual oscillation at ~0.3% of the weight, so the calibration (and therefore the
# loss) settles cleanly. If you cut TRAIN_N so far that an epoch is only a few steps, raise
# this to 1.0; the trade is convergence speed against how tightly the loss can settle.
#
# This only affects loss and calibration. AUROC is a property of the feature itself, so it
# is unchanged by the learning rate — see the diagnostic cell at the end.
# cfg.learning_rate is deliberately left alone, for Surya in notebook 2.
# ---------------------------------------------------------------------------
baseline_lr = 0.1
print(f"baseline_lr = {baseline_lr} (config's learning_rate = {cfg.learning_rate}, "
      f"kept for Surya)")

# Baseline 1: constant probability — never touches 'ts', so no preprocess_fn needed.
#lit_model_constant = WaveLightningModule(
#    model_constant, make_metrics(), lr=baseline_lr, batch_size=batch_size,
#)

# Baseline 2: running difference — needs the signum-log-space inverse transform wired up
# so WaveLightningModule applies it before every model call.
preprocess_fn = partial(
    destandardize_channels,
    channel_order=cfg.data.channels,
    scalers=scalers,
)
lit_model_running_diff = WaveLightningModule(
    model_running_diff, make_metrics(), lr=baseline_lr, batch_size=batch_size,
    preprocess_fn=preprocess_fn,
)


baseline_lr = 0.1 (config's learning_rate = 0.01, kept for Surya)


## Logging

In order to properly compare experiments against each other, it is very useful to log evaluation metrics in a place where they can be compared against other training runs.  In this workshop we will use Weights and Biases (WandB). 

The first time you run WandB in a machine it will ask you to login to WandB.  You should have received an invitation to our project.  In order to login you must:

- Select option 2 (existing account).   In VScode the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VSCode

In [25]:
# WandB is now authenticated via `wandb login` (stored in ~/.netrc as lloverasdiego /
# surya-ws2), so WandbLogger below logs live — no offline/disabled override needed.

In [28]:
if wandb.run is not None:
    print(f"finishing live wandb run: {wandb.run.name}")
    wandb.finish()
else:
    print("no live wandb run")

finishing live wandb run: baseline_experiment_vector


epoch,▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆▆▆▆████████
train_loss,▄▃▄▆▄▅▅▄▅▅▅▇▅▄▅▅▁█▄▄▃▅▅▂▂▇▅▅▅▅▄▅▅▅▅▅▄▄▄▃
train_loss_bce,▃▂▃▃▃▄▃▃▃▃▃▃▃▃▂▁█▃▃▂▅▃▃▃▂▃▃▃▃▄▄▃▃▄▃▃▃▃▂▂
train_metric_accuracy,▅▅▅█▅▅▅█▅▅▅▁▅▁▁▁██▁▁▅▅▅▁▅▁█▅█▅▅▅█▁▁▅▅▁▅█
trainer/global_step,▁▁▁▁▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
val_loss,▅█▁▁
val_loss_bce,▅█▁▁
val_metric_accuracy,▁▁▁▁
val_metric_auroc,█▁▆▇
epoch,3
train_loss,0.60682


In [29]:
project_name = cfg.wandb_project
run_name = "baseline_experiment_vector"  # give your run a descriptive name

wandb_logger = WandbLogger(
    entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
    project=project_name,
    name=run_name,
    log_model=False,
    save_dir="./wandb/wandb_tmp",
)

csv_logger = CSVLogger("runs", name=project_name)


## Initialize trainer

With the loggers done, now the trainer needs to be defined.  The trainer defines several properties of your training run. Here we define:

- The max number of epochs (one epoch represents your model seeing your entire training dataset).
- Define where the training run will take place (auto uses the GPU if possible, if not, CPU).
- The loggers.
- The callbacks (here we save the model with the lowest validation loss).
- Logging frequency (because we are working with a small dataset it needs to be small).

In [30]:
max_epochs = 20

# -------------------------------------------------------------------------
# Trainer
# -------------------------------------------------------------------------
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices="auto",
    logger=[wandb_logger, csv_logger],
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        )
    ],
    log_every_n_steps=2,
)

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Fit the models

Finally we fit each baseline. We pass its Lightning module, and the shared dataloaders
(both baselines see the same train/val split, so their validation performance is directly
comparable).

In [31]:
trainer.fit(lit_model_running_diff, train_data_loader, val_data_loader)



LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name  | Type                           | Params | Mode  | FLOPs
-------------------------------------------------------------------------
0 | model | RunningDifferenceLogisticModel | 6      | train | 0    
-------------------------------------------------------------------------
6         Trainable params
0         Non-trainable params
6         Total params
0.000     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'

## Diagnostic: is the feature informative at all?

The two runs above tell you what the models *learned*. For a two-parameter model that is
monotone in a single scalar, that is less than it sounds: the ordering it induces on the
validation samples is fixed the moment the weight has a sign, so **AUROC is a property of
the feature, not of the training**. Only the loss and the threshold are learned.

So the loss curves cannot answer the question that matters — *does
`mean(|pool32(AIA193 now − prev)|)` separate wave from no-wave at all?* The cell below
answers it directly, with no training, on the validation split.
